In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, BooleanType
from pyspark.sql import DataFrame
from delta.tables import DeltaTable
import logging

In [0]:
# tables_to_drop = [
#         "silver_base_events",
#         "silver_pass",
#         "silver_ball_receipt",
#         "silver_shot",
#         "silver_shot_freeze_frame",
#         "silver_carry",
#         "silver_defensive_action",
#         "silver_match_lineups",
#         "silver_match_player_positions",
#         "silver_360_frames"
#     ]

# for table in tables_to_drop:
#     spark.sql(f"DROP TABLE IF EXISTS workspace.football_project.{table}")
#     print(f"Dropped: workspace.football_project.{table}")

In [0]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

logger = logging.getLogger("football_pipeline")

##### Prep Bronze

In [0]:
# prep bronze table

df_bronze_events = spark.table("workspace.football_project.bronze_events")
df_bronze_lineups = spark.table("workspace.football_project.bronze_lineups")
df_bronze_360 = spark.table("workspace.football_project.bronze_three_sixty")

In [0]:
# df_bronze_events.show(10)

##### Functions

In [0]:
### Common Spine Columns:

def get_common_columns() -> list:

    return[
        # Keys
        F.col("competition_id"),
        F.col("season_id"),
        F.col("match_id"),
        F.col("index"),
        F.col("id").alias("event_id"),
        
        # Chrono
        F.col("period"),
        F.col("timestamp"),
        F.col("minute"),
        F.col("second"),

        # Events & Players
        F.col("type.name").alias("event_type"),
        F.col("player.id").alias("player_id"),
        F.col("player.name").alias("player_name"),
        F.col("team.id").alias("team_id"),
        F.col("team.name").alias("team_name"),

        # Context
        F.col("location").alias("start_location"),
        F.get(F.col("location"),0).alias("start_x"),
        F.get(F.col("location"),1).alias("start_y"),
        F.col("possession").alias("possession_id"),
        F.col("possession_team.id").alias("possession_team_id"),
        F.col("possession_team.name").alias("possession_team_name"),
        F.coalesce(F.col("under_pressure"), F.lit(False)).alias("is_under_pressure"),
        F.col("play_pattern.name").alias("play_pattern_name"),
        F.coalesce(F.col("duration"), F.lit(0.0)).alias("event_duration"),
        F.col("related_events").alias("related_events"),

        # Meta
        F.col("_file_path").alias("_bronze_file_path"),
]
   

In [0]:
### Flatten Function

def flatten_dataframe(df: DataFrame) -> DataFrame:
    
    complex_fields = [f for f in df.schema.fields if isinstance(f.dataType, StructType)]
    
    if not complex_fields:
        return df
   
    select_expr = []
    for f in df.schema.fields:
        if isinstance(f.dataType, StructType):
            for child in f.dataType.fields:
                select_expr.append(f" {f.name}.{child.name} AS {f.name}_{child.name} ")
        else:
            select_expr.append(f.name)

    return flatten_dataframe(df.selectExpr(*select_expr)) 

In [0]:
### Split End Location [x,y,z] Function

def end_location_split(df: DataFrame) -> DataFrame:
    exprs = []
    for col_name in df.columns:
        if col_name.endswith('end_location'):
            prefix = col_name.split('end_location')[0]
            exprs.append(F.col(col_name))
            exprs.append(F.get(F.col(col_name), 0).alias(f"{prefix}end_x" ))
            exprs.append(F.get(F.col(col_name), 1).alias(f"{prefix}end_y" ))
            exprs.append(F.coalesce(F.get(F.col(col_name), 2), F.lit(0.0)).alias(f"{prefix}end_z"))                     
        else:
            exprs.append(F.col(col_name))

    return df.select(*exprs)

In [0]:
### Fill NA Function

def fill_na_bool_columns(df: DataFrame) -> DataFrame:
    bool_cols = [cols.name for cols in df.schema.fields if isinstance(cols.dataType, BooleanType)]

    df_fill_na = df.fillna(False, subset=bool_cols)
    return df_fill_na

In [0]:
### Clean Glitch Location : -5 < X < 125, -5 < Y < 85 Function

def clean_glitch_location(df: DataFrame, min_glitch_x: float = -5.0, max_glitch_x: float = 125.0,
                           min_glitch_y: float = -5.0, max_glitch_y: float = 85.0) -> DataFrame:

    exprs = []
    for col_name in df.columns:
        if col_name.endswith('_x'):
            exprs.append(F.when(F.col(col_name).isNull(), F.lit(None))
                         .when((F.col(col_name) < min_glitch_x ) | (F.col(col_name) > max_glitch_x), F.lit(None))
                         .otherwise(F.col(col_name))
                         .alias(col_name))
            
        elif col_name.endswith('_y'):
            exprs.append(F.when(F.col(col_name).isNull(), F.lit(None))
                         .when((F.col(col_name) < min_glitch_y ) | (F.col(col_name) > max_glitch_y), F.lit(None))
                         .otherwise(F.col(col_name))
                         .alias(col_name))

        else:
            exprs.append(F.col(col_name))

    return df.select(*exprs)


In [0]:
### SILVER: Standard Cleaning Pipeline

def clean_silver_dataframe(df: DataFrame) -> DataFrame:
    """
    Standard cleaning suite for all Silver event tables:
    1. Flattens all nested StructType columns.
    2. Fills null booleans with False.
    3. Splits end_location into end_x, end_y, end_z (if exists).
    """

    return( 
        df
        .transform(flatten_dataframe)
        .transform(end_location_split)
        .transform(clean_glitch_location)
        .transform(fill_na_bool_columns)
    )

In [0]:
### Rearrnage Metadata

def finalize_silver_dataframe(df: DataFrame) -> DataFrame:

    df_metadata = (
        df
        .withColumn("_silver_inserted_at", F.current_timestamp())
        .withColumn("_silver_updated_at", F.current_timestamp())
        .withColumn("_source_pipeline", F.lit("nb_bronze_to_silver_events"))
    )

    common_cols = [c for c in df_metadata.columns if not c.startswith("_")]
    metadata_cols = [c for c in df_metadata.columns if c.startswith("_")]

    return df_metadata.select(*common_cols, *metadata_cols)

In [0]:
### UPSERT Function 
 
def upsert_delta_table(df: DataFrame, table_name: str, join_keys: list|dict, partition_cols: str|list) -> None:

    """
    Universal Delta Lake Upsert (MERGE INTO).

    :param df: Incoming source DataFrame
    :param table_name: Target Delta table (e.g. 'football_project.silver_shots')
    :param join_keys:
        - list: When names match, e.g. ['matchId', 'eventId']
        - dict: When names differ, e.g. {'target_col': 'source_col'} -> {'eventId': 'event_uuid'}
    :param partition_col: Column to partition by on first creation (default: 'matchId')
    :return: None
    """
    
    if isinstance(join_keys, dict):
        join_condition = " AND ".join([f"target.{target} = source.{source}" for target,source in join_keys.items()])
    elif isinstance(join_keys, list):
        join_condition = " AND ".join([f"target.{k} = source.{k}" for k in join_keys])
    else:
        raise ValueError("join_keys must be a list or dict")

    if spark.catalog.tableExists(table_name):
        (
            DeltaTable.forName(spark, table_name)
            .alias("target")
            .merge(df.alias("source"), join_condition)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        logger.info(f"Successfully upserted data into: {table_name}")
    else:
        (
            df.write.format("delta")
            .mode("overwrite")
            .partitionBy(partition_cols)
            .saveAsTable(table_name)
        )
        logger.info(f" Created new Delta table: {table_name} (partitioned by '{partition_cols}')")

## Run

#### [1] SILVER: Base Event

In [0]:
df_silver_base_events = (
    df_bronze_events
    .select(*get_common_columns())
    .transform(clean_silver_dataframe)
    .transform(finalize_silver_dataframe)
)

In [0]:
### UPSERT : Silver base events
 
upsert_delta_table(
    df = df_silver_base_events, 
    table_name = "football_project.silver_base_events", 
    join_keys = ["match_id", "event_id"],
    partition_cols = "competition_id"
)

#### [2] Categorized Event Table 
(Passes, Recipient, Shots, Carry, Defendsive Actions)

###### Passes

In [0]:
### SELECT: Pass

df_silver_pass = (
    df_bronze_events
    .filter(F.col("type.name") == "Pass")
    .select(
        *get_common_columns(),
        F.col("pass")
    )
    .transform(clean_silver_dataframe)
    .drop("pass_end_z")
    .withColumn("is_pass_complete", F.col("pass_outcome_name").isNull())
    .fillna({"pass_outcome_name": "Complete"})
    .withColumn("is_pass_to_final_third",
            (F.col("start_x") < 80.0) & (F.col("pass_end_x") >= 80.0))  
    .withColumn("is_pass_into_penalty_box",
            (F.col("pass_end_x") >= 102.0) & (F.col("pass_end_y") >= 18.0) & (F.col("pass_end_y") <= 62.0))
    .transform(finalize_silver_dataframe)
)

In [0]:
# df_silver_pass.filter(
#     (F.col("pass_goal_assist") == True) & 
#     (F.col("teamName")=="Japan") &
#     (F.col("pass_outcome_name") == "Complete")
#     ).show()

In [0]:
# df_silver_pass.show(10)

In [0]:
### UPSERT: Pass

upsert_delta_table(
    df = df_silver_pass, 
    table_name = "football_project.silver_pass", 
    join_keys = ["match_id", "event_id"], 
    partition_cols = "competition_id") 

###### Receipt

In [0]:
### SELECT: Receipt
 
df_silver_ball_receipt = (
        df_bronze_events
        .filter(F.col("type.name") == "Ball Receipt*")
        .select(
            *get_common_columns(),
            F.col("ball_receipt").alias("receipt")
        )
        .transform(clean_silver_dataframe)
        .withColumn("is_receipt_complete", F.col("receipt_outcome_name").isNull())
        .fillna({"receipt_outcome_name": "Complete"})
        .transform(finalize_silver_dataframe)
    )

In [0]:
#### UPSERT: receipt

upsert_delta_table(
        df=df_silver_ball_receipt,
        table_name="football_project.silver_ball_receipt",
        join_keys=["match_id", "event_id"],
        partition_cols="competition_id" 
    )

###### Shot

In [0]:
### SELECT: Shot

df_silver_shot = (
    df_bronze_events
    .filter(F.col("type.name") == "Shot")
    .select(
        *get_common_columns(),
        F.col("shot")
    )
    .transform(clean_silver_dataframe)
    # Mark for Pipeline 3.0 : Shot on target isin ["Post", "Goal", "Blocked", "Saved", "Saveed To Post"]
    .withColumn("is_goal", F.col("shot_outcome_name") == "Goal")
    .withColumn("shot_distance", 
                        F.sqrt(F.pow(F.col("shot_end_x") - F.col("start_x"), 2) 
                              + F.pow(F.col("shot_end_y") - F.col("start_y"),2))
                       )
    )

In [0]:
### CLEAN: Shot Table
df_silver_shot_drop_freeze_frame = (df_silver_shot
                                    .drop(F.col("shot_freeze_frame"))
                                    .transform(finalize_silver_dataframe)
                                    )

In [0]:
### UPSERT: Shot

upsert_delta_table(
    df = df_silver_shot_drop_freeze_frame, 
    table_name = "football_project.silver_shot", 
    join_keys=["match_id", "event_id"],
    partition_cols="competition_id" 
)

In [0]:
# df_silver_shot_drop_freeze_frame.show(10)

In [0]:
# SELECT: Freeze Frame
 
df_silver_shot_freeze_frame = (
        df_silver_shot
        .withColumn("ff", F.explode_outer("shot_freeze_frame"))
        .select(
            # Keys
            F.col("competition_id"),
            F.col("match_id"),
            F.col("index"),
            F.col("event_id"),
            
            # Freezze Frame
            F.col("ff.player.id").alias("freeze_player_id"),
            F.col("ff.player.name").alias("freeze_player_name"),
            F.col("ff.position.id").alias("freeze_position_id"),
            F.col("ff.position.name").alias("freeze_position_name"),
            F.col("ff.teammate").alias("is_teammate"),
            (F.col("ff.position.id") == 1).alias("is_keeper"),
            F.get(F.col("ff.location"), 0).alias("freeze_x"),
            F.get(F.col("ff.location"), 1).alias("freeze_y")
        )
        .transform(finalize_silver_dataframe)
)

In [0]:
### UPSERT: Shot Freeze Frame

upsert_delta_table(
    df = df_silver_shot_freeze_frame, 
    table_name = "football_project.silver_shot_freeze_frame", 
    join_keys = ["match_id", "event_id","freeze_player_id"], 
    partition_cols = "competition_id"
) 

###### Carry

In [0]:
### SELECT: Carry

df_silver_carry = (
    df_bronze_events
    .filter(F.col("type.name") == "Carry")
    .select(
        *get_common_columns(),
        F.col("carry")
    )
    .transform(clean_silver_dataframe)
    .withColumn("carry_distance", 
                F.sqrt(
                    F.pow(F.col("carry_end_x") - F.col("start_x"),2) +
                    F.pow(F.col("carry_end_y") - F.col("start_y"),2)
                    )
                )
    .drop(F.col("carry_end_z"))
    .transform(finalize_silver_dataframe)
)

In [0]:
# df_silver_carry.show(10, truncate=False)

In [0]:
### UPSERT: Carry

upsert_delta_table(
    df = df_silver_carry, 
    table_name = "football_project.silver_carry", 
    join_keys = ["match_id", "event_id"], 
    partition_cols = "competition_id") 

###### Defensive Actions

In [0]:
### SELECT: Defendsive Actions

DEF_ACTIONS = ["50/50", "Block", "Clearance", "Duel", "Foul Committed", "Interception", "Pressure"]

df_silver_defend = (
    df_bronze_events
    .filter(F.col("type.name").isin(DEF_ACTIONS))
    .select(
        *get_common_columns(),
        F.col("duel"),
        F.col("interception"),
        F.col("clearance"),
        F.col("block"),
        F.col("50_50").alias("fifty_fifty"),
        F.col("foul_committed")
    )
    .transform(clean_silver_dataframe)
    .transform(finalize_silver_dataframe)
)

In [0]:
# df_silver_defend.show(10)

In [0]:
### UPSERT: Defend

upsert_delta_table(
    df=df_silver_defend,
    table_name="football_project.silver_defensive_action",
    join_keys=["match_id", "event_id"],
    partition_cols = "competition_id"
)

#### [3] Match Lineups

In [0]:
### SELECT: Match Lineups

df_silver_match_lineups = (
    df_bronze_lineups
    .select(
        F.col("competition_id"),
        F.col("match_id"),
        F.col("team_id"),
        F.col("team_name"),
        F.explode_outer("lineup").alias("player"),
        F.col("_file_path"),
    )
    .select(
        F.col("competition_id"),
        F.col("match_id"),
        F.col("team_id"),
        F.col("team_name"),
        F.col("player.player_id").alias("player_id"),
        F.col("player.player_name").alias("player_name"),
        F.col("player.player_nickname").alias("player_nickname"),
        F.col("player.jersey_number").alias("jersey_number"),
        F.col("player.country.name").alias("country_name"),
        (F.size(F.col("player.positions")) > 0).alias("has_played"),
        (F.coalesce(F.get(F.col("player.positions"), 0).getField("start_reason"), F.lit("")) == "Starting XI").alias("is_starter"),
        F.get(F.col("player.positions"), 0).getField("position_id").alias("starting_position_id"),
        F.get(F.col("player.positions"), 0).getField("position").alias("starting_position_name"),
        F.col("_file_path")
    )
    .transform(clean_silver_dataframe)
    .transform(finalize_silver_dataframe)
)

In [0]:
### UPSERT: Match Lineups

upsert_delta_table(
    df=df_silver_match_lineups,
    table_name="football_project.silver_match_lineups",
    join_keys=["match_id", "player_id"],
    partition_cols = "competition_id"
)

In [0]:
### SELECT:  Match Player Position
BASE_COLS = ["competition_id", "match_id", "team_id", "team_name", "player_id", "player_name", "jersey_number"]

df_silver_match_player_positions = (
    df_bronze_lineups
    .select(
      F.col("competition_id"),
      F.col("match_id"),
      F.col("team_id"),
      F.col("team_name"),
      F.explode_outer("lineup").alias("player"))
    .filter(F.size(F.col("player.positions")) > 0)
    .select(
         F.col("competition_id"),
        F.col("match_id"),
        F.col("team_id"),
        F.col("team_name"),
        F.col("player.player_id").alias("player_id"),
        F.col("player.player_name").alias("player_name"),
        F.col("player.jersey_number").alias("jersey_number"),
        F.explode_outer(F.col("player.positions")).alias("pp")
        )
    .select(
        *BASE_COLS,
        F.col("pp.position_id").alias("position_id"),
        F.col("pp.position").alias("position_name"),
        F.col("pp.from").alias("from_time"),             
        F.col("pp.to").alias("to_time"),                    
        F.col("pp.from_period").alias("from_period"),         
        F.col("pp.to_period").alias("to_period"),  
        F.col("pp.start_reason").alias("start_reason"),
        F.col("pp.end_reason").alias("end_reason")
        )
    .transform(clean_silver_dataframe)
    .dropDuplicates(["match_id", "player_id", "position_id", "from_time", "from_period"])
    .transform(finalize_silver_dataframe)
)

In [0]:
### UPSERT: Match Player Position

upsert_delta_table(
    df=df_silver_match_player_positions,
    table_name="football_project.silver_match_player_positions",
    join_keys=["match_id", "player_id", "position_id", "from_time", "from_period"],
    partition_cols = "competition_id"
)

#### [4] 360

In [0]:
df_silver_360 = (
    df_bronze_360
    .select(
        F.col("competition_id"),
        F.col("match_id"),
        F.col("event_uuid").alias("event_id"),
        F.col("visible_area"),
        F.posexplode_outer("freeze_frame").alias("player_track_id", "ff"),
        F.col("_file_path")
    )
    .select(
        F.col("competition_id"),
        F.col("match_id"),
        F.col("event_id"),
        F.col("visible_area"),
        F.col("player_track_id"),
        F.col("ff.teammate").alias("is_teammate"),
        F.col("ff.actor").alias("is_actor"),
        F.col("ff.keeper").alias("is_keeper"),
        F.get(F.col("ff.location"), 0).alias("player_x"),
        F.get(F.col("ff.location"), 1).alias("player_y"),
        F.col("_file_path")
    )
    .transform(clean_glitch_location)
    .transform(fill_na_bool_columns)
    .transform(finalize_silver_dataframe)
)

In [0]:
### UPSERT: 360

upsert_delta_table(
    df=df_silver_360,
    table_name="football_project.silver_360_frames",
    join_keys=["match_id", "event_id", "player_track_id"],
    partition_cols="competition_id"
)